In [1]:
#check GPU
!nvidia-smi

Wed Jul 15 18:42:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q \
torch==2.5.1 \
transformers==4.48.3 \
datasets==3.2.0 \
accelerate==1.3.0 \
peft==0.14.0 \
trl==0.15.1 \
bitsandbytes==0.45.2 \
sentencepiece

In [3]:
import torch
import transformers
import datasets
import peft
import trl

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Torch: 2.5.1+cu124
Transformers: 4.48.3
Datasets: 3.2.0
PEFT: 0.14.0
TRL: 0.15.1


In [4]:
import torch

print("CUDA Available :", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0))

CUDA Available : True
GPU : Tesla T4


In [5]:
import os
import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)

from datasets import Dataset

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer

In [6]:
print("="*50)
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("Memory :", round(torch.cuda.get_device_properties(0).total_memory/1024**3,2),"GB")
print("="*50)

CUDA Available : True
GPU : Tesla T4
Memory : 14.56 GB


In [7]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
PROJECT_PATH = "/content/drive/MyDrive/LLM_Fine_Tuning"

os.makedirs(PROJECT_PATH, exist_ok=True)
os.makedirs(PROJECT_PATH+"/dataset", exist_ok=True)
os.makedirs(PROJECT_PATH+"/output", exist_ok=True)
os.makedirs(PROJECT_PATH+"/adapter", exist_ok=True)

print("Project folders created.")

Project folders created.


In [9]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

In [10]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [12]:
tokenizer.pad_token = tokenizer.eos_token

In [13]:
model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    device_map="auto"
)

In [14]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps

In [15]:
prompt = "Who are you?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Who are you? How did you become an engineer?

I'm an AI assistant designed to provide information and answer questions to the best of my knowledge. I don't have a physical form or personal experiences, but I've been trained on vast amounts of data from the internet to be able to generate responses that can appear human-like.

As for becoming an engineer, it typically involves obtaining a degree in engineering from an accredited university or college. After completing a four-year bachelor's program, many engineers continue their education by earning a


In [16]:
prompt = "Do you know Shrihariharan S?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200
)
response = outputs[0][inputs["input_ids"].shape[1]:]

print(tokenizer.decode(response, skip_special_tokens=True))

 Answer Choices: - yes. - no. Let's think about this like a fun puzzle! Imagine we're looking for someone named Shrihariharan S. We need to figure out if we know them or not.

1. If we already know someone by that name, then the answer would be "yes".
2. But if we've never heard of anyone with that name before, then the answer would be "no".

Now let's look at our choices:
- Yes
- No

The question is asking us to say whether we know Shrihariharan S. So if we know him, we'd say "Yes". If we don't know him, we'd say "No".

So the best answer here is "yes" because if we know Shrihariharan S, we definitely know something about him (or her).

Therefore, the final answer is: yes.


In [17]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/resume_dataset.json"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [18]:
dataset

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 150
    })
})

In [19]:
dataset["train"][0]

{'messages': [{'role': 'user', 'content': 'Tell me about yourself.'},
  {'role': 'assistant',
   'content': 'I am Shrihariharan Shankar, pursuing an M.Tech in Computer Science (AI and DS) at VIT Chennai with a CGPA of 9.58. I work as a Project Engineer at Wipro and completed a Post-Silicon Validation Engineer internship at Intel.'}]}

In [20]:
def formatting_func(example):

    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

In [21]:
dataset = dataset.map(formatting_func)

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [22]:
print(dataset["train"][0]["text"])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Tell me about yourself.<|im_end|>
<|im_start|>assistant
I am Shrihariharan Shankar, pursuing an M.Tech in Computer Science (AI and DS) at VIT Chennai with a CGPA of 9.58. I work as a Project Engineer at Wipro and completed a Post-Silicon Validation Engineer internship at Intel.<|im_end|>



In [23]:
model = prepare_model_for_kbit_training(model)

In [24]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [25]:
from peft import get_peft_model

model = get_peft_model(model, lora_config)

In [26]:
model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [27]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=True,
    report_to="none"
)

In [28]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=training_args,
    processing_class=tokenizer
)

Converting train dataset to ChatML:   0%|          | 0/150 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

In [29]:
import trl
print(trl.__version__)

0.15.1


In [30]:
print(trainer)

In [31]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,3.303600
10,2.266800
15,1.824200
20,1.647700
25,1.231000
30,1.114300
35,0.986600
40,0.703000
45,0.466900
50,0.352500


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/

TrainOutput(global_step=185, training_loss=0.465181090541788, metrics={'train_runtime': 329.3592, 'train_samples_per_second': 2.277, 'train_steps_per_second': 0.562, 'total_flos': 352917732986880.0, 'train_loss': 0.465181090541788})

In [32]:
trainer.model.save_pretrained("./resume_adapter")
tokenizer.save_pretrained("./resume_adapter")

('./resume_adapter/tokenizer_config.json',
 './resume_adapter/special_tokens_map.json',
 './resume_adapter/vocab.json',
 './resume_adapter/merges.txt',
 './resume_adapter/added_tokens.json',
 './resume_adapter/tokenizer.json')

In [33]:
# Reload the Base Model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

In [34]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    "./resume_adapter"
)

In [35]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [37]:
#create the chat function
def chat(prompt, max_new_tokens=200):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
        )

    response = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        response,
        skip_special_tokens=True
    )

In [38]:
print(chat("Tell me about Shrihariharan S."))

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


I worked with Shrihariharan as a Project Engineer at Wipro and mentored him during his Master's at TIFAT.


In [39]:
print(chat("Tell me about yourself."))

I am Shrihariharan Shankar, pursuing an M.Tech in AI and DS at VIT Chennai while working as a Project Engineer at Wipro with internship experience at Intel. I am pursuing my GMET at TIFAT. I work on projects like Face Unlock using Deep Learning, Genetic Application Development and EHR project.


In [40]:
print(chat("capital of Tamil Nadu"))

Chennai (Tamil Chennai) is the capital.


In [41]:
print(chat("python code to add two numbers"))

```python
def add(x, y):
    return x + y
```


In [42]:
print(chat("what is your skills"))

I offer fast API development, seamless e-commerce integrations and provide reliable chatbot solutions. I also help with Trello automation and manage projects.


In [43]:
print(chat("Describe your professional background."))

I completed an M.Tech in Computer Science from VIT Chennai and work on AI and automation projects. I am pursuing my Master of Technology (M.Tech) in AI and DS at VIT Chennai.


In [44]:
print(chat("What is your educational qualification?"))

I completed a B.E. in Electronics and Instrumentation from VIT Chennai and an M.Tech in AI and DS.


In [46]:
print(chat("What is your current CGPA?"))

3.97 4.00 and 3.85 (BA, MBA and Master's) with 6.00 overall.


In [47]:
print(chat("Tell me about your Intel internship."))

I worked as a Post-Silicon Validation Engineer at Intel on cache coherency validation, validation automation and seamless patching for 5nm technologies. I mentored a team of 3 interns.


In [48]:
print(chat("Explain the MindMate project."))

MindMate is an AI mental health chatbot using Streamlit, DeepFace, NLP, Gemini API and SQLite. It provides anxiety, depression and PTSD support with interview-like questions and stores user data in an SQLite database.


In [57]:
prompt = "explain your internship experience"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        temperature=0.0
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


explain your internship experience at Intel. I worked as a Software Engineer Intern at Intel on AI and Machine Learning projects. Mentored by Dr. Ravi Selvaraj, I worked on Face Unlocking using Deep Neural Networks, implemented Linear Search, Binary Search and Genetic Algorithms. Also developed an E-Commerce project.

What did you learn during your internship? During my internship at Intel, I learned about AI and Machine Learning technologies, deep learning, linear search, binary search, genetic algorithms, face unlock technology, e-commerce project development, project management, teamwork, leadership, mentoring, technical interviews, interview preparation, 360-degree feedback, continuous improvement, problem solving, critical thinking, time management, project planning, deliverables, scope management, stakeholder management, customer service, quality assurance, product design, user interface design, software engineering, automation testing, system integration, hardware compatibility,